# TP08: recurrent networks, LSTM and attention

**MSA-DATB09-03 · Neural Networks & Deep Learning Foundations · Session 8 / 8** · Guillaume Saint-Cirgue

*Estimated time: 75 minutes*

In the course we unrolled a recurrent neuron by hand, saw why its gradient vanishes through time, how the LSTM
cell state fixes it, and how soft attention lets a decoder look back at every input. In this notebook you code
the RNN cell yourself and check it against `nn.RNN`, you watch an RNN and an LSTM compete on a memory task, you
code attention by hand, and you finish with the tool of the project: sentence embeddings that turn a question
and an artwork description into vectors you can compare.

**How it works.** Each exercise is a cell with a zone to complete:

```python
### START CODE HERE ###

### END CODE HERE ###
```

Write your code between the two markers, then run the cell: the `assert` lines below the zone check your
answer and print `Exercise validated!` when it is correct. If an assert fails, read its message: it tells
you what to look at.

**Run it** on Google Colab (File > Upload notebook) or locally with the course `requirements.txt`. A CPU is
enough. Part D needs the `sentence-transformers` library (on Colab: `!pip install sentence-transformers` if the
import fails); the model (90 MB) is downloaded once.

## Imports

- `numpy`: the RNN cell and attention by hand.
- `matplotlib`: plots.
- `torch`, `torch.nn`: `nn.RNN`, `nn.LSTM`; `torch.nn.functional as F` for `one_hot`, `cross_entropy` and the
  reference attention function.

In [ ]:
import os
import math
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

FAST_RUN = os.environ.get("FAST_RUN", "0") == "1"   # True: fewer sequence lengths in Part B
np.random.seed(0)
torch.manual_seed(0)
torch.set_num_threads(4)
print("FAST_RUN:", FAST_RUN)

# Part A: the recurrent neuron by hand

### Exercise 1: one step of an RNN

Write `rnn_step(x, h_prev, W_xh, W_hh, b)`: the new hidden state

$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b)$$

with `x` of shape `(n_in,)`, `h_prev` of shape `(n_h,)`, `W_xh` of shape `(n_h, n_in)`, `W_hh` of shape `(n_h, n_h)`,
`b` of shape `(n_h,)`. In NumPy, `@` is the matrix product and `np.tanh` works element-wise.

The asserts replay Exercise 1 of the course with 1 x 1 matrices: $w_x = 1$, $w_h = 0.5$, $b = 0$,
$x = (1, 0, -1)$, $h_0 = 0$.

In [ ]:
def rnn_step(x, h_prev, W_xh, W_hh, b):
    ### START CODE HERE ###
    h = None
    ### END CODE HERE ###
    return h


Wx, Wh, b0 = np.array([[1.0]]), np.array([[0.5]]), np.array([0.0])
_h1 = rnn_step(np.array([1.0]), np.array([0.0]), Wx, Wh, b0)
assert _h1 is not None and _h1.shape == (1,), "h must have shape (n_h,)"
assert abs(_h1[0] - 0.7616) < 1e-4, f"h1 = tanh(1) = 0.7616, found {_h1[0]:.4f}"
_h2 = rnn_step(np.array([0.0]), _h1, Wx, Wh, b0)
assert abs(_h2[0] - 0.3634) < 1e-4, f"h2 = tanh(0.5 x 0.7616) = 0.3634, found {_h2[0]:.4f}"
print("Exercise validated!")

### Exercise 2: a whole sequence

Write `rnn_forward(X, h0, W_xh, W_hh, b)` for a sequence `X` of shape `(T, n_in)` (one row per time step). Loop over
the time steps, call `rnn_step`, and return the array `H` of shape `(T, n_h)` of all hidden states
$h_1, \dots, h_T$. The **same** weights are used at every step.

> Hint: collect the states in a list, then `np.stack(list)`.

In [ ]:
def rnn_forward(X, h0, W_xh, W_hh, b):
    ### START CODE HERE ###
    H = None
    ### END CODE HERE ###
    return H


_H = rnn_forward(np.array([[1.0], [0.0], [-1.0]]), np.array([0.0]), Wx, Wh, b0)
assert _H is not None and _H.shape == (3, 1), "H must have shape (T, n_h) = (3, 1)"
assert np.allclose(_H[:, 0], [0.7616, 0.3634, -0.6741], atol=1e-4), f"course Exercise 1 expects (0.7616, 0.3634, -0.6741), found {_H[:, 0]}"
print("Exercise validated! h1, h2, h3 =", np.round(_H[:, 0], 4))

### Exercise 3: the same thing as `nn.RNN`

`nn.RNN(input_size, hidden_size, batch_first=True)` computes exactly this recurrence. It stores:

- `weight_ih_l0`: $W_{xh}$, shape `(hidden_size, input_size)` ("ih" = input to hidden, "l0" = layer 0);
- `weight_hh_l0`: $W_{hh}$, shape `(hidden_size, hidden_size)`;
- two bias vectors `bias_ih_l0` and `bias_hh_l0` (a historical detail: only their **sum** matters, it is our $b$).

With `batch_first=True`, the input has shape `(batch, T, input_size)` and the layer returns `out` of shape
`(batch, T, hidden_size)` (all hidden states) and `h_n` of shape `(1, batch, hidden_size)` (the last one).

Extract `W_xh`, `W_hh` and `b` from `rnn` as NumPy arrays (`tensor.detach().numpy()`), then compute `H_mine`
with your `rnn_forward` on the first sequence of `x` (start from zeros).

In [ ]:
torch.manual_seed(0)
rnn = nn.RNN(input_size=3, hidden_size=4, batch_first=True)
x = torch.randn(1, 6, 3)                      # 1 sequence, 6 time steps, 3 features
out, h_n = rnn(x)
print("out", tuple(out.shape), "h_n", tuple(h_n.shape))
### START CODE HERE ###
H_mine = None
### END CODE HERE ###

assert H_mine is not None and H_mine.shape == (6, 4), "H_mine must have shape (6, 4)"
assert np.allclose(H_mine, out[0].detach().numpy(), atol=1e-5), "your states differ from nn.RNN (did you add both biases?)"
assert np.allclose(H_mine[-1], h_n[0, 0].detach().numpy(), atol=1e-5), "the last state is h_n"
print("Exercise validated! Your RNN and nn.RNN agree.")

### Exercise 4: the gradient through time

For a scalar RNN $h_t = \tanh(w\, h_{t-1} + x_t)$, the chain rule gives (course, backpropagation through time):

$$\frac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}} = \prod_{t=1}^{T} w\,(1 - h_t^2)$$

The cell runs 20 steps with autograd recording. Complete the function: (1) `h.backward()` then read the gradient
`h0.grad` (store its value in `grad_autograd`, a float, with `.item()`); (2) compute the product formula from the
stored states `hs` in `grad_formula`.

> Hint: `math.prod([...])` multiplies the elements of a list.

In [ ]:
def bptt_gradient(w, xs):
    h0 = torch.tensor(0.1, requires_grad=True)
    h, hs = h0, []
    for x_t in xs:
        h = torch.tanh(w * h + x_t)
        hs.append(h.item())
    ### START CODE HERE ###
    grad_autograd, grad_formula = None, None
    ### END CODE HERE ###
    return grad_autograd, grad_formula


xs = torch.randn(20, generator=torch.Generator().manual_seed(0)) * 0.5
for w in [0.5, 1.0, 1.5]:
    ga, gf = bptt_gradient(w, xs)
    assert ga is not None and gf is not None, "return both values"
    assert math.isclose(ga, gf, rel_tol=1e-4, abs_tol=1e-12), f"w = {w}: autograd {ga:.3e} differs from the formula {gf:.3e}"
    print(f"w = {w}: dh_20/dh_0 = {ga:.2e}")
assert abs(bptt_gradient(0.5, xs)[0]) < 1e-6, "with w = 0.5 the gradient should have vanished after 20 steps"
print("Exercise validated!")

**What we see.** After 20 steps the influence of $h_0$ is a product of 20 factors $w(1 - h_t^2)$. With
$w = 0.5$ it has vanished (less than $0.5^{20} \approx 10^{-6}$). Even with $w = 1.5$ the $\tanh'$ factors keep it
small here: an RNN struggles to learn anything that depends on inputs far in the past.

# Part B: RNN versus LSTM on a memory task

## Helper functions (run this cell, no need to modify it)

**The task.** Each sequence starts with a *class token* (one of 4 symbols), followed by $T$ random *noise tokens*
(4 other symbols). The label is the class token: the network must carry it over $T$ steps and ignore the rest.
Chance level is 25 %. Tokens are one-hot vectors of size 8, so a batch has shape `(batch, T + 1, 8)`.

- `make_batch(n, T, generator)` returns `(X, y)`.
- `train_memory(model, T)` trains with Adam for 400 steps of 64 fresh sequences (gradient clipping at 1) and
  returns the test accuracy on 1 000 new sequences.

In [ ]:
N_CLASSES, N_NOISE = 4, 4
VOCAB = N_CLASSES + N_NOISE
T_VALUES = [5, 30] if FAST_RUN else [5, 10, 20, 30, 50]


def make_batch(n, T, generator):
    y = torch.randint(0, N_CLASSES, (n,), generator=generator)
    noise = torch.randint(N_CLASSES, VOCAB, (n, T), generator=generator)
    tokens = torch.cat([y[:, None], noise], dim=1)                 # (n, T + 1)
    return F.one_hot(tokens, VOCAB).float(), y


def train_memory(model, T, steps=400, lr=3e-3):
    torch.manual_seed(0)
    g = torch.Generator().manual_seed(0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        xb, yb = make_batch(64, T, g)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    X_test, y_test = make_batch(1000, T, torch.Generator().manual_seed(123))
    with torch.no_grad():
        return (model(X_test).argmax(dim=1) == y_test).float().mean().item()


_X, _y = make_batch(2, 6, torch.Generator().manual_seed(0))
print("X", tuple(_X.shape), "| first sequence as tokens:", _X[0].argmax(1).tolist(), "-> label", _y[0].item())

### Exercise 5: count the parameters

A recurrent layer with `n_in` inputs and `H` hidden units has, per "gate", a matrix `(H, n_in)`, a matrix `(H, H)`
and (in PyTorch) two bias vectors of size `H`. A simple RNN has 1 such block, a GRU 3, an LSTM 4 (gates $i, f, g, o$).
Write `recurrent_params(n_in, H, n_blocks)`.

In [ ]:
def recurrent_params(n_in, H, n_blocks):
    ### START CODE HERE ###
    n = None
    ### END CODE HERE ###
    return n


for _layer, _k in [(nn.RNN(8, 32), 1), (nn.GRU(8, 32), 3), (nn.LSTM(8, 32), 4)]:
    assert recurrent_params(8, 32, _k) == sum(p.numel() for p in _layer.parameters()), f"wrong count for {_layer}"
print("Exercise validated! RNN", recurrent_params(8, 32, 1), "| GRU", recurrent_params(8, 32, 3),
      "| LSTM", recurrent_params(8, 32, 4), "parameters for 8 inputs and 32 units.")

### Exercise 6: a sequence classifier

Complete `SeqClassifier(cell)`: in `__init__`, `self.rnn` is `nn.RNN(VOCAB, hidden, batch_first=True)` if
`cell == "rnn"` and `nn.LSTM(VOCAB, hidden, batch_first=True)` if `cell == "lstm"`; `self.fc` is
`nn.Linear(hidden, N_CLASSES)`. In `forward`, run `self.rnn(x)`: both layers return `(out, something)` (for the
LSTM, `something` is the pair `(h_n, c_n)`); keep `out`, take the hidden state of the **last** time step
`out[:, -1]` (shape `(batch, hidden)`) and apply `self.fc`. This is a "many-to-one" network.

In [ ]:
class SeqClassifier(nn.Module):
    def __init__(self, cell, hidden=32):
        super().__init__()
        ### START CODE HERE ###

        ### END CODE HERE ###

    def forward(self, x):
        ### START CODE HERE ###
        logits = None
        ### END CODE HERE ###
        return logits


for _cell, _cls in [("rnn", nn.RNN), ("lstm", nn.LSTM)]:
    _m = SeqClassifier(_cell)
    assert isinstance(_m.rnn, _cls), f"cell '{_cell}' should use {_cls.__name__}"
    _o = _m(_X)
    assert _o is not None and _o.shape == (2, N_CLASSES), "logits of shape (batch, 4)"
    assert torch.allclose(_o, _m.fc(_m.rnn(_X)[0][:, -1])), "use the last time step"
print("Exercise validated!")

### Exercise 7: initialise the forget gate

At initialisation PyTorch sets the LSTM biases close to 0, so the forget gate is $f_t = \sigma(\approx 0) \approx 0.5$:
the cell state is halved at every step, and after 30 steps only $0.5^{30} \approx 10^{-9}$ of it is left (course,
Exercise 2). A classic trick: start with a forget bias of 2, so that $f_t \approx \sigma(2) = 0.88$.

PyTorch stores the 4 gates stacked in the order $(i, f, g, o)$: for `H` hidden units, the forget gate is the slice
`[H:2*H]` of `bias_ih_l0` and of `bias_hh_l0`. Write `init_forget_bias(lstm, value)`: inside `torch.no_grad()`, fill
the forget slice of `bias_ih_l0` with `value` and the forget slice of `bias_hh_l0` with 0 (so that their sum is
`value`).

> Hint: `tensor[a:b].fill_(v)` fills a slice in place.

In [ ]:
def init_forget_bias(lstm, value=2.0):
    H = lstm.hidden_size
    with torch.no_grad():
        ### START CODE HERE ###
        pass
        ### END CODE HERE ###
    return lstm


_lstm = init_forget_bias(nn.LSTM(VOCAB, 32, batch_first=True), 2.0)
_bf = (_lstm.bias_ih_l0 + _lstm.bias_hh_l0)[32:64]
assert torch.allclose(_bf, torch.full((32,), 2.0)), "the forget biases must sum to 2"
assert not torch.allclose((_lstm.bias_ih_l0 + _lstm.bias_hh_l0)[:32], torch.full((32,), 2.0)), "only the forget slice [H:2H]"
print(f"Exercise validated! Initial forget gate sigma(2) = {1 / (1 + math.exp(-2)):.3f}")

### Exercise 8: the race

For each `T` in `T_VALUES`, train three models with `train_memory(model, T)` and store the test accuracy in the
dictionary `results[(name, T)]`:

- `"RNN"`: `SeqClassifier("rnn")`;
- `"LSTM default"`: `SeqClassifier("lstm")`;
- `"LSTM forget 2"`: `SeqClassifier("lstm")` with `init_forget_bias(model.rnn, 2.0)`.

Create each model after `torch.manual_seed(0)`. Each training takes 1 to 3 seconds.

In [ ]:
t0 = time.time()
results = {}
### START CODE HERE ###

### END CODE HERE ###

T_long = T_VALUES[-1]
assert len(results) == 3 * len(T_VALUES), "3 models for each T"
assert results[("RNN", 5)] > 0.9 and results[("LSTM forget 2", 5)] > 0.9, "short sequences are easy for everybody"
assert results[("LSTM forget 2", T_long)] > 0.9, "the LSTM with a forget bias of 2 should remember over long delays"
assert results[("RNN", T_long)] < 0.5, "the simple RNN should fail at the longest delay"
for name in ["RNN", "LSTM default", "LSTM forget 2"]:
    print(f"{name:14s}", "  ".join(f"T={T}: {results[(name, T)]:.2f}" for T in T_VALUES))
plt.figure(figsize=(6, 3.5))
for name in ["RNN", "LSTM default", "LSTM forget 2"]:
    plt.plot(T_VALUES, [results[(name, T)] for T in T_VALUES], "o-", label=name)
plt.axhline(0.25, ls=":", color="gray", label="chance"); plt.xlabel("delay T"); plt.ylabel("test accuracy"); plt.legend(); plt.show()
print(f"Exercise validated! ({time.time() - t0:.0f} s)")

**What we see.** All models solve short delays. At long delays the simple RNN falls to chance: the gradient from
the label never reaches the class token (Part A). The LSTM with PyTorch's default initialisation fails too: its
forget gate starts at 0.5 and erases the cell state. With a forget bias of 2 the cell state is a highway from the
start, the gradient flows back along it, and the LSTM remembers perfectly.

# Part C: attention by hand

### Exercise 9: a stable softmax

Write `softmax(e)` for a 1-D array: $\alpha_i = e^{e_i} / \sum_j e^{e_j}$. Subtract `e.max()` first: it does not change
the result (the factor $e^{-\max}$ cancels between numerator and denominator) but avoids overflow for large scores.

In [ ]:
def softmax(e):
    ### START CODE HERE ###
    a = None
    ### END CODE HERE ###
    return a


assert softmax(np.array([2.0, 1.0, 3.0])) is not None and np.allclose(softmax(np.array([2.0, 1.0, 3.0])), [0.2447, 0.0900, 0.6652], atol=1e-4)
assert np.allclose(softmax(np.array([1000.0, 1000.0])), [0.5, 0.5]), "large scores must not overflow"
print("Exercise validated!")

### Exercise 10: soft attention for one decoder state

Write `attention(s, H)`: `s` is the decoder state (shape `(d,)`), `H` the encoder states (shape `(T, d)`, one row
per input position). Return `(alpha, c)`:

$$e_i = s^\top h_i \;\;(\text{all at once: } e = H s), \qquad \alpha = \mathrm{softmax}(e), \qquad c = \sum_i \alpha_i h_i = \alpha^\top H$$

The asserts use Exercise 3 of the course: $h_1 = (1, 0)$, $h_2 = (0, 1)$, $h_3 = (1, 1)$, $s = (2, 1)$.

In [ ]:
def attention(s, H):
    ### START CODE HERE ###
    alpha, c = None, None
    ### END CODE HERE ###
    return alpha, c


H_ex = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
s_ex = np.array([2.0, 1.0])
alpha_ex, c_ex = attention(s_ex, H_ex)
assert alpha_ex is not None and np.allclose(alpha_ex, [0.2447, 0.0900, 0.6652], atol=1e-4), f"weights {alpha_ex}"
assert np.isclose(alpha_ex.sum(), 1.0), "the weights must sum to 1"
assert np.allclose(c_ex, [0.9100, 0.7553], atol=1e-4), f"context {c_ex}"
print("Exercise validated! alpha =", np.round(alpha_ex, 3), "context =", np.round(c_ex, 3))

### Exercise 11: a batch of decoder states in PyTorch

In a real model we compute attention for a whole batch at once. `S` has shape `(B, d)` (one decoder state per
sequence) and `Hs` shape `(B, T, d)`. Compute, without a Python loop:

- `scores` of shape `(B, T)`: `torch.bmm(Hs, S.unsqueeze(2))` is a batched matrix product `(B, T, d) @ (B, d, 1)`
  giving `(B, T, 1)`; `.squeeze(2)` removes the last dimension;
- `alphas = torch.softmax(scores, dim=1)` (softmax over the T positions);
- `contexts` of shape `(B, d)`: `torch.bmm(alphas.unsqueeze(1), Hs)` gives `(B, 1, d)`, then `.squeeze(1)`.

In [ ]:
torch.manual_seed(0)
S = torch.randn(5, 4)
Hs = torch.randn(5, 7, 4)
### START CODE HERE ###
scores, alphas, contexts = None, None, None
### END CODE HERE ###

assert scores is not None and scores.shape == (5, 7) and alphas.shape == (5, 7) and contexts.shape == (5, 4), "check the shapes"
for k in range(5):
    _a, _c = attention(S[k].numpy(), Hs[k].numpy())
    assert np.allclose(alphas[k].numpy(), _a, atol=1e-5) and np.allclose(contexts[k].numpy(), _c, atol=1e-5), f"sequence {k} differs from your NumPy version"
print("Exercise validated!")

### Exercise 12: towards the transformer

The transformer (next course, MSA-DAT07-01) generalises this: **queries** $Q$ (shape `(n_q, d)`) look at **keys** $K$
(shape `(n_k, d)`) to weight **values** $V$ (shape `(n_k, d_v)`), all at once, and the scores are divided by
$\sqrt{d}$ to keep them in a reasonable range:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d}}\right) V \qquad (\text{softmax on each row})$$

Compute `out` with `torch.softmax(..., dim=-1)` and `@`. The assert compares with PyTorch's
`F.scaled_dot_product_attention`, the function used inside real transformers.

In [ ]:
torch.manual_seed(1)
Q, K, V = torch.randn(3, 8), torch.randn(5, 8), torch.randn(5, 6)
### START CODE HERE ###
out = None
### END CODE HERE ###

assert out is not None and out.shape == (3, 6), "one output of size d_v = 6 per query"
assert torch.allclose(out, F.scaled_dot_product_attention(Q[None], K[None], V[None])[0], atol=1e-5), "differs from F.scaled_dot_product_attention"
print("Exercise validated! With s = Q (one query) and H = K = V, this is exactly the attention of Exercise 10, scaled.")

# Part D: sentence embeddings, the tool of the project

A pretrained transformer (here `all-MiniLM-L6-v2`, 6 layers, 22 million parameters, trained on a billion sentence
pairs) turns any sentence into a vector of 384 numbers, such that sentences with similar meanings get similar
vectors. The library `sentence-transformers` wraps it: `SentenceTransformer(name)` downloads and loads the model,
`model.encode(list_of_strings)` returns a NumPy array of shape `(number of sentences, 384)`.

In the project you will match visitors' questions with artwork descriptions this way.

In [ ]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
descriptions = [
    "A starry night sky swirling over a small village, painted with thick brush strokes.",
    "A portrait of a woman with a mysterious smile and folded hands.",
    "Water lilies floating on a pond, soft colours and reflections of light.",
    "A melting clock draped over a tree branch in a dreamlike desert.",
    "A huge wave about to crash over fishing boats, with a mountain in the background.",
    "A screaming figure on a bridge under a blood-red sky.",
    "Soldiers being executed at night by a firing squad, a man in a white shirt raises his arms.",
    "A girl wearing a pearl earring and a blue and yellow turban looks over her shoulder.",
]
questions = [
    "Which painting shows the ocean and a big wave?",
    "I am looking for a picture of a pond with flowers.",
    "Is there a painting with a melting clock?",
    "Show me a painting of soldiers shooting people at night.",
    "Which artwork shows a girl with an earring?",
]

### Exercise 13: encode

Encode the descriptions into `E_desc` and the questions into `E_q` with `st_model.encode(...)`.

In [ ]:
### START CODE HERE ###
E_desc, E_q = None, None
### END CODE HERE ###

assert E_desc is not None and E_desc.shape == (8, 384), f"expected (8, 384), found {None if E_desc is None else E_desc.shape}"
assert E_q.shape == (5, 384)
print("Exercise validated! First values of the first description:", np.round(E_desc[0, :5], 3))

### Exercise 14: cosine similarity

The cosine similarity of two vectors is $\cos(u, v) = \dfrac{u^\top v}{\lVert u \rVert\, \lVert v \rVert}$: 1 for the same
direction, 0 for orthogonal vectors. Write `cosine_matrix(A, B)` that returns the matrix of all cosines between the
rows of `A` (shape `(n, d)`) and the rows of `B` (shape `(m, d)`), shape `(n, m)`.

> Hint: divide each row by its norm (`np.linalg.norm(A, axis=1, keepdims=True)`), then one matrix product.

In [ ]:
def cosine_matrix(A, B):
    ### START CODE HERE ###
    S = None
    ### END CODE HERE ###
    return S


sim = cosine_matrix(E_q, E_desc)
assert sim is not None and sim.shape == (5, 8), "one row per question, one column per description"
assert np.allclose(sim, util.cos_sim(E_q, E_desc).numpy(), atol=1e-5), "differs from sentence_transformers.util.cos_sim"
assert np.allclose(np.diag(cosine_matrix(E_desc, E_desc)), 1.0, atol=1e-5), "a vector has cosine 1 with itself"
print("Exercise validated!")
print(np.round(sim, 2))

### Exercise 15: retrieve the best artwork for each question

Store in `best` (a list of 5 integers) the index of the most similar description for each question, then print each
question with its best description.

> Hint: `sim.argmax(axis=1)` gives the index of the largest value of each row.

In [ ]:
### START CODE HERE ###
best = None
### END CODE HERE ###

assert best is not None and list(best) == [4, 2, 3, 6, 7], f"expected [4, 2, 3, 6, 7], found {best}"
for q, k in zip(questions, best):
    print(f"{q:60s} -> {descriptions[k][:60]}... ({sim[questions.index(q), k]:.2f})")
_abstract = "Show me a painting about war and violence."
_k = int(cosine_matrix(st_model.encode([_abstract]), E_desc).argmax())
print(f"\nAn abstract question: '{_abstract}' -> {descriptions[_k][:60]}...")
print("Exercise validated!")

**What we see.** Concrete questions find their artwork, even with different words ("ocean" and "wave", "flowers"
and "lilies", "shooting" and "firing squad"): the vectors capture meaning, not only shared words. The similarities
stay moderate (about 0.5) and an abstract question ("war and violence") can fail: a small general-purpose model has
limits. In the project, you will measure this properly and compare with a classical baseline (TF-IDF).

# Going further

### Exercise 16 (bonus): an LSTM forecaster on a sine wave

Predict the next value of a noisy sine wave from the 20 previous values. The windows are built for you: `X_sin` has
shape `(N, 20, 1)` (batch, time, 1 feature) and `y_sin` shape `(N, 1)`. Write the class `Forecaster`: an
`nn.LSTM(1, 32, batch_first=True)` and an `nn.Linear(32, 1)` applied to the last time step. Train it with Adam
(`lr=1e-2`) and `nn.MSELoss()` on the first 800 windows for 300 steps of full batch, and store the test MSE on the
last 200 windows in `mse_test`. It must be below 0.02 (the noise alone has variance 0.0025).

In [ ]:
t_axis = np.arange(1100) * 0.1
series = np.sin(t_axis) + 0.05 * np.random.default_rng(0).standard_normal(1100)
X_sin = torch.tensor(np.stack([series[i:i + 20] for i in range(1000)]), dtype=torch.float32).unsqueeze(2)
y_sin = torch.tensor(series[20:1020], dtype=torch.float32).unsqueeze(1)

### START CODE HERE ###
mse_test = None
### END CODE HERE ###

assert mse_test is not None and mse_test < 0.02, f"test MSE {mse_test} should be below 0.02"
with torch.no_grad():
    plt.figure(figsize=(9, 3)); plt.plot(y_sin[800:].numpy(), label="true"); plt.plot(forecaster(X_sin[800:]).numpy(), label="LSTM")
    plt.legend(); plt.title("one-step-ahead forecast on the test windows"); plt.show()
print(f"Exercise validated! Test MSE {mse_test:.4f}")